In [ ]:
import ee
import geemap
import time

# ---------------------------------------------------------
# Authenticate and initialize Earth Engine
# ---------------------------------------------------------
ee.Authenticate(scopes=['https://www.googleapis.com/auth/drive'])
ee.Initialize(project='geospatial-study')

# ---------------------------------------------------------
# 1. Define ROI polygon
# ---------------------------------------------------------
roi = ee.Geometry.Polygon([
    [
        [-111.639219, 37.185186],  # NW
        [-110.935586, 37.185186],  # NE
        [-110.935586, 36.919714],  # SE
        [-111.639219, 36.919714],  # SW
        [-111.639219, 37.185186]   # Close polygon
    ]
])

# ---------------------------------------------------------
# 2. Landsat 5 TM (May 1985)
# ---------------------------------------------------------
collection_1985 = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
    .filterBounds(roi)
    .filterDate("1985-06-01", "1985-06-30")
)

landsat_1985 = (
    collection_1985
    .mosaic()
    .select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    .clip(roi)
)

date_1985 = collection_1985.first().get("DATE_ACQUIRED").getInfo()
print("1985 image date:", date_1985)

# ---------------------------------------------------------
# 3. Landsat 9 OLI/TIRS (June 2025)
# -----------------------------------------06----------------
collection_2025 = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterBounds(roi)
    .filterDate("2025-06-01", "2025-06-30")
)

landsat_2025 = (
    collection_2025
    .mosaic()
    .select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
    .clip(roi)
)

date_2025 = collection_2025.first().get("DATE_ACQUIRED").getInfo()
print("2025 image date:", date_2025)

# ---------------------------------------------------------
# 4. Export both rasters to Google Drive (GeoTIFF)
# ---------------------------------------------------------
task1 = ee.batch.Export.image.toDrive(
    image=landsat_1985,
    description='landsat_1985_june_drive_export',
    fileNamePrefix='landsat_1985_june',
    scale=30,
    region=roi,
    maxPixels=1e13
)
task1.start()

task2 = ee.batch.Export.image.toDrive(
    image=landsat_2025,
    description='landsat_2025_june_drive_export',
    fileNamePrefix='landsat_2025_june',
    scale=30,
    region=roi,
    maxPixels=1e13
)
task2.start()

print("Export tasks started. Checking task status...")

time.sleep(3)
for task in ee.batch.Task.list():
    print(task.status())










1985 image date: 1985-06-14
2025 image date: 2025-06-04
Export tasks started. Checking task status...
{'state': 'READY', 'description': 'landsat_2025_june_drive_export', 'priority': 100, 'creation_timestamp_ms': 1772646547621, 'update_timestamp_ms': 1772646547621, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'AGPJLZ57XWT2OZPDOQLLQ2EF', 'name': 'projects/geospatial-study/operations/AGPJLZ57XWT2OZPDOQLLQ2EF'}
{'state': 'READY', 'description': 'landsat_1985_june_drive_export', 'priority': 100, 'creation_timestamp_ms': 1772646547176, 'update_timestamp_ms': 1772646547176, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'DZBR4TKWJTEZLSHWUUALS2GJ', 'name': 'projects/geospatial-study/operations/DZBR4TKWJTEZLSHWUUALS2GJ'}
{'state': 'COMPLETED', 'description': 'landsat_2025_june_drive_export', 'priority': 100, 'creation_timestamp_ms': 1772646154454, 'update_timestamp_ms': 1772646401192, 'start_timestamp_ms': 1772646332029, 'task_type': 'EXPORT_IMAGE', 'destination_uris